# Exercice 01 – Corrigé

Version corrigée du notebook `01_comparaison_modeles.ipynb`. Le code est volontairement simple ; il existe des façons plus élégantes de faire chaque étape.

In [ ]:
import glob
import time

import matplotlib.pyplot as plt
import polars as pl
import requests
from openai import OpenAI

## Étape 0 – Lancer le serveur (dans un terminal)

```bash
llama-server \
  -m /chemin/vers/modele_A.gguf \
  -c 8192 -np 1 -fa on -b 2048 -ub 512 -ngl 99 \
  --host 127.0.0.1 --port 8080
```

> **Tip** : notez quelque part le nom exact du modèle et sa quantisation (par exemple `Qwen3-8B-Q4_K_M`). Vous en aurez besoin pour nommer le fichier de résultats.

In [ ]:
BASE_URL = "http://127.0.0.1:8080"
client = OpenAI(base_url=f"{BASE_URL}/v1", api_key="pas-besoin")

# Nom court du modèle actuellement servi : à modifier à chaque relance du serveur
MODEL_NAME = "modele_A"

# Vérification que le serveur répond
print(requests.get(f"{BASE_URL}/health").json())
print([m.id for m in client.models.list().data])

## Étape 1 – Charger les questions

In [ ]:
def charger_questions(chemin: str) -> pl.DataFrame:
    lignes = []
    with open(chemin, encoding="utf-8") as f:
        for ligne in f:
            ligne = ligne.strip()
            if ligne.startswith("|"):
                cellules = [c.strip() for c in ligne.strip("|").split("|")]
                lignes.append(cellules)

    entete = lignes[0]          # ['id', 'categorie', 'type', 'question', 'reponse_attendue']
    donnees = lignes[2:]        # nous sautons l'en-tête et la ligne de séparation |----|
    df = pl.DataFrame(donnees, schema=entete, orient="row")
    return df.with_columns(pl.col("id").cast(pl.Int64))

questions = charger_questions("01_questions.md")
print(len(questions), "questions")
print(questions["type"].value_counts())
questions.head()

## Étape 2 – Une fonction pour poser une question

In [ ]:
SYSTEM_PROMPT = "Réponds en français, de façon précise et concise."

def poser(question: str) -> dict:
    r = client.chat.completions.create(
        model="local",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": question},
        ],
        temperature=0,
        max_tokens=500,
        # Désactive le mode <think> de Qwen3. Ignoré par les modèles qui ne le connaissent pas.
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    t = r.model_extra["timings"]
    return {
        "reponse": r.choices[0].message.content.strip(),
        "completion_tokens": r.usage.completion_tokens,
        "prompt_tokens": r.usage.prompt_tokens,
        "tok_par_s": round(t["predicted_per_second"], 1),
        "duree_ms": round(t["prompt_ms"] + t["predicted_ms"]),
        "finish_reason": r.choices[0].finish_reason,
    }

poser("Quelle est la capitale de l'Australie ?")

## Étape 3 – Boucle sur les 50 questions

In [ ]:
resultats = []
debut = time.perf_counter()

for i, q in enumerate(questions.iter_rows(named=True), start=1):
    res = poser(q["question"])
    resultats.append({
        "modele": MODEL_NAME,
        "id": q["id"],
        "categorie": q["categorie"],
        "type": q["type"],
        "question": q["question"],
        "reponse_attendue": q["reponse_attendue"],
        **res,   # fusionne le dictionnaire renvoyé par poser()
    })
    print(f"{i:2d}/50  {res['duree_ms']:5d} ms  {res['tok_par_s']:5.1f} tok/s  |  {q['question'][:60]}")

duree_totale = time.perf_counter() - debut
df = pl.DataFrame(resultats)
df.write_csv(f"resultats_{MODEL_NAME}.csv")
print(f"\nTerminé en {duree_totale:.0f} s → resultats_{MODEL_NAME}.csv")

## Étape 4 – Changer de modèle

`Ctrl+C` dans le terminal, relancer avec le modèle B, changer `MODEL_NAME = "modele_B"` dans le setup, ré-exécuter setup + étape 3.

## Étape 5 – Évaluation automatique des questions fermées

In [ ]:
fichiers = sorted(glob.glob("resultats_*.csv"))
print("Fichiers trouvés :", fichiers)
tous = pl.concat([pl.read_csv(f) for f in fichiers], how="vertical_relaxed")

# correct = True/False pour les questions fermées, null pour les ouvertes
tous = tous.with_columns(
    pl.when(pl.col("type") == "fermee")
    .then(
        pl.col("reponse").str.to_lowercase()
        .str.contains(pl.col("reponse_attendue").cast(pl.String).str.to_lowercase(), literal=True)
    )
    .otherwise(None)
    .alias("correct")
)

print("Taux de bonnes réponses (questions fermées) :")
print(tous.group_by("modele").agg(pl.col("correct").mean().round(2)).sort("modele"))

print("\nPar catégorie :")
print(
    tous.filter(pl.col("type") == "fermee")
    .pivot(on="modele", index="categorie", values="correct", aggregate_function="mean")
    .with_columns(pl.col(pl.Float64).round(2))
    .sort("categorie")
)

In [ ]:
# Les cas marqués faux : à lire pour repérer les limites de la vérification par `in`
faux = tous.filter(pl.col("correct") == False).select("modele", "id", "question", "reponse_attendue", "reponse")
pl.Config.set_fmt_str_lengths(120)
faux

## Étape 6 – Vitesse, longueur, questions ouvertes

In [ ]:
stats = tous.group_by("modele").agg(
    tok_par_s_moyen=pl.col("tok_par_s").mean(),
    tokens_moyens=pl.col("completion_tokens").mean(),
    duree_totale_s=pl.col("duree_ms").sum() / 1000,
    reponses_coupees=(pl.col("finish_reason") == "length").sum(),
).sort("modele")
print(stats)

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))
axes[0].bar(stats["modele"], stats["tok_par_s_moyen"]); axes[0].set_title("Vitesse (tok/s)")
axes[1].bar(stats["modele"], stats["tokens_moyens"]);   axes[1].set_title("Longueur moyenne (tokens)")
axes[2].bar(stats["modele"], stats["duree_totale_s"]);  axes[2].set_title("Durée totale (s)")
plt.tight_layout()
plt.show()

In [ ]:
# Lecture côte à côte de quelques questions ouvertes
for qid in [17, 34, 46]:
    sous = tous.filter(pl.col("id") == qid)
    print("=" * 100)
    print(f"Q{qid} : {sous[0, 'question']}")
    print(f"Attendu : {sous[0, 'reponse_attendue']}")
    for ligne in sous.iter_rows(named=True):
        print("-" * 100)
        print(f"[{ligne['modele']}]  ({ligne['completion_tokens']} tokens)\n")
        print(ligne["reponse"])
    print()

## Éléments de réponse aux questions de réflexion

1. **Justesse vs vitesse** : en général le modèle le plus gros (ou le moins quantisé) est plus juste et plus lent. Mais un modèle bavard n'est pas forcément plus juste : regardez `completion_tokens` par rapport à `correct`.
2. **Divergences** : souvent sur `maths` et `logique`. Les petits modèles font des erreurs d'arithmétique et de comptage ; la culture générale est plus homogène.
3. **Compter des lettres** : le modèle ne voit pas des lettres mais des tokens (`ferroviaire` peut être 2 ou 3 tokens). Il n'a pas d'accès direct aux caractères, il doit « se souvenir » de l'orthographe.
4. **Limites du `in`** : question 6, le modèle peut répondre « dix » au lieu de « 10 » (faux négatif) ; question 28, « 60 » est contenu dans « 600 » ou « 1960 » (faux positif). Améliorations : normaliser les nombres écrits en lettres, chercher le mot entier avec une expression régulière (`\b60\b`), ou accepter plusieurs réponses attendues.
5. **`temperature=0`** : rend la génération déterministe, donc la comparaison porte sur les modèles et pas sur le hasard. Avec `temperature=1`, deux exécutions du même modèle donneraient des réponses différentes ; il faudrait répéter chaque question plusieurs fois et moyenner.